# 🏥 ChakraModel Cloud GPU Evaluation Notebook (Kaggle/Colab)

This notebook contains the complete environment to load the ChakraModel architecture, load all pretrained weights, and run cross-verification benchmarks using a cloud GPU like Kaggle T4x2 or Google Colab.

## Setup Instructions for Kaggle:
1. **Upload your local `chakramodel` folder as a Kaggle Dataset**. 
   - Create a ZIP file of your local `chakramodel` directory (ensure `weights/best.pt` is inside).
   - In Kaggle, click **Add Input** -> **Upload a Dataset**.
   - Name it `chakramodel-data`.
   - Your files will then be available at `/kaggle/input/chakramodel-data/`.
2. In the Kaggle Notebook Settings (right sidebar), turn on **Accelerator: GPU P100 or T4x2**.
3. Set `REPO_PATH` in the cell below to point to your dataset (e.g. `/kaggle/input/chakramodel-data/chakramodel`).


In [3]:
!pip install -q ultralytics

import sys
import shutil
from pathlib import Path

# Rest of your code continues below...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.9 MB/s eta 0:00:00


In [2]:
!pip install ultralytics opencv-python-headless pandas numpy torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 613.8 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.1 MB/s eta 0:00:00


In [3]:
import sys
import os
from pathlib import Path

def find_repo_root():
    """Automatically finds the dataset folder in Kaggle by searching for the 'src' directory"""
    if os.path.exists('/kaggle/input'):
        for path in Path('/kaggle/input').rglob('src'):
            if path.is_dir():
                return str(path.parent)
    return '.' # Fallback if running locally

REPO_PATH = find_repo_root()

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f"Working directory set to: {os.getcwd()}")

Working directory set to: /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle


In [4]:
import torch
from ultralytics import YOLO
from src.pranet_segmenter import PraNetSegmenter
from src.paris_classifier import ParisClassifier
from src.benchmark_kvasir import run_benchmark
from pathlib import Path

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
CUDA Available: True
Device Name: Tesla T4


## Download Evaluation Dataset
Downloading the Kvasir-SEG benchmark dataset to verify the cross-dataset metrics and test inference.

In [5]:
from src.download_kvasir import download_kvasir_seg

# This will download the dataset to REPO_PATH/data/kvasir-seg 
# (If it fails due to network, it falls back to synthetic data so pipelines can still run)
root_dir = Path(REPO_PATH)

# Note: In Kaggle /kaggle/input is read-only. If the dataset needs to be written to,
# we must point root_dir to /kaggle/working. But download_kvasir_seg defaults to root_dir / "data".
if "/kaggle/input" in str(root_dir):
    working_dir = Path("/kaggle/working")
    dataset_path = download_kvasir_seg(working_dir)
else:
    working_dir = root_dir
    dataset_path = download_kvasir_seg(root_dir)

print(f"Dataset ready at: {dataset_path}")

  Progress: 100.0%  (44MB / 44MB)
[EXTRACT] kvasir-seg.zip...
[OK] kvasir-seg ready: 1000 images
Dataset ready at: /kaggle/working/data/kvasir-seg


## Load the Weights
Loading the trained YOLO weights and initializing the ChakraModel PraNet architecture.

In [6]:
# Determine the path to YOLO best weights
weight_candidates = [
    root_dir / "weights/best.pt",
    root_dir / "outputs/polyp_yolov8x/weights/best.pt",
    root_dir / "best.pt",
]

best_weight_path = None
for c in weight_candidates:
    if c.exists():
        best_weight_path = c
        break

if best_weight_path:
    print(f"Loading YOLO weights from: {best_weight_path}")
    yolo_model = YOLO(str(best_weight_path))
    if torch.cuda.is_available():
        yolo_model.to('cuda')
else:
    print("WARNING: YOLO best.pt weights not found in standard directories. Make sure weights are uploaded.")

# Initialize PraNet
print("Initializing PraNet Segmenter...")
pranet = PraNetSegmenter(device='cuda' if torch.cuda.is_available() else 'cpu')

# Initialize Paris Classifier
print("Initializing Paris Classifier...")
paris = ParisClassifier()
print("All ChakraModel components loaded successfully!")

Loading YOLO weights from: /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/weights/best.pt
Initializing PraNet Segmenter...
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 209MB/s]


[INFO] PraNetSegmenter: Loaded weights from /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/weights/pranet_kvasir_best.pth
Initializing Paris Classifier...
All ChakraModel components loaded successfully!


## Run Benchmark Evaluation
Run the benchmark script directly on Kvasir-SEG to calculate Dice, mIoU, Sensitivity, Specificity, and FPS. This will automatically cross-verify the results from local training.

In [7]:
import sys
import shutil
from pathlib import Path

# 1. Create a writable src directory in /kaggle/working
working_src = Path("/kaggle/working/src")
working_src.mkdir(parents=True, exist_ok=True)

# 2. Point to the original read-only file and the new writable location
orig_bmk = Path(REPO_PATH) / "src" / "benchmark_kvasir.py"
patched_bmk = working_src / "benchmark_kvasir.py"

# 3. Read the original, patch the resolution bug, and write it to the writable folder
code = orig_bmk.read_text()
code = code.replace("segmenter = PraNetSegmenter()", "segmenter = PraNetSegmenter(img_size=(352, 352))")
patched_bmk.write_text(code)

# 4. Create an empty __init__.py so Python recognizes it as a module
(working_src / "__init__.py").touch()

# 5. Force Python to clear its memory of the old script and look in the writable folder FIRST
if "src.benchmark_kvasir" in sys.modules:
    del sys.modules["src.benchmark_kvasir"]

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

print("✅ Successfully patched benchmark_kvasir.py in the writable /kaggle/working directory!")

✅ Successfully patched benchmark_kvasir.py in the writable /kaggle/working directory!


In [8]:
out_dir = working_dir / "outputs" / "eval_kaggle"
out_dir.mkdir(parents=True, exist_ok=True)

print("Running full benchmark on Kvasir-SEG...")
# Temporarily adjust dataset paths since Kaggle /kaggle/input is read-only and 
# download_kvasir_seg puts it in working_dir.
import src.benchmark_kvasir as bmk
# Override the dataset root to point to the working dir where we downloaded it
metrics = bmk.run_benchmark("kvasir-seg", working_dir, out_dir)

import json
print("\nBenchmark Results Summary:")
print(json.dumps({
    "Dice (DSC)": metrics['dice'],
    "mIoU": metrics['iou'],
    "Sensitivity": metrics['sensitivity'],
    "FPS": metrics['fps']
}, indent=2))

Running full benchmark on Kvasir-SEG...

[BENCHMARK] ChakraModel PraNet on kvasir-seg
  Images: 1000
  Device: auto (CUDA if available)
[INFO] PraNetSegmenter: Loaded weights from /kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/weights/pranet_kvasir_best.pth
  [  50/1000] Running Dice: 0.4412 | 10.8ms/frame
  [ 100/1000] Running Dice: 0.4039 | 12.5ms/frame
  [ 150/1000] Running Dice: 0.4477 | 11.7ms/frame
  [ 200/1000] Running Dice: 0.4822 | 11.0ms/frame
  [ 250/1000] Running Dice: 0.4764 | 11.5ms/frame
  [ 300/1000] Running Dice: 0.4997 | 10.5ms/frame
  [ 350/1000] Running Dice: 0.5096 | 12.5ms/frame
  [ 400/1000] Running Dice: 0.5003 | 11.5ms/frame
  [ 450/1000] Running Dice: 0.5010 | 11.7ms/frame
  [ 500/1000] Running Dice: 0.4998 | 10.8ms/frame
  [ 550/1000] Running Dice: 0.5083 | 11.1ms/frame
  [ 600/1000] Running Dice: 0.5072 | 13.6ms/frame
  [ 650/1000] Running Dice: 0.5034 | 10.5ms/frame
  [ 700/1000] Running Dice: 0.5127 | 11.2ms/frame
  [ 750/1000] Running Dice: 0

## Test on Live Video (Optional)
If you have `test_input.mp4` uploaded with your dataset, you can process it through the pipeline just like the Gradio app.

In [1]:
import sys
import shutil
from pathlib import Path

# 1. Create a writable directory in /kaggle/working
working_src = Path("/kaggle/working/src")
working_src.mkdir(parents=True, exist_ok=True)

# 2. Copy src contents to writable directory
orig_src = Path("/kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/src")
for file in orig_src.glob("*.py"):
    shutil.copy2(file, working_src / file.name)

# 3. Add /kaggle/working to Python path before /kaggle/input
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# 4. Clear cached imports
if "src.app" in sys.modules:
    del sys.modules["src.app"]
if "app" in sys.modules:
    del sys.modules["app"]

print("✅ Copy complete. You can now edit /kaggle/working/src/app.py")

✅ Copy complete. You can now edit /kaggle/working/src/app.py


In [13]:
import os
print("Folders in /kaggle/input:")
print(os.listdir('/kaggle/input'))

Folders in /kaggle/input:
['datasets']


In [14]:
import os

print("Contents of /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files[:3]:  # Print first 3 files
        print(f"{subindent}{f}")

Contents of /kaggle/input:
input/
    datasets/
        gokulrocky/
            polypdataset-gokul/
                1_14.avi
                1_34.avi
                1_6.avi
            chakramodel/
                kaggle_bundle/
                    README.md
                    test_input.mp4
                    requirements.txt
                    weights/
                        pranet_kvasir_best.pth
                        best.pt
                    notebooks/
                        Combo6_ChakraTransformer.ipynb
                        Combo6_ChakraTransformer.py
                        Combo1_ChakraNet_Focal.ipynb
                        deprecated/
                            combo4_diffusion_aug_kaggle.py
                            Combo4_DiffusionAug_ChakraNet.py
                            Combo4_DiffusionAug_ChakraNet.ipynb
                        data/
                            filtered_synthetic_polyps/
                                images/
                        

In [9]:
import sys
sys.path.append('/kaggle/input/datasets/gokulrocky/polypdataset-gokul')


import sys
import os
import shutil
from pathlib import Path

# 1. Register your uploaded source code folder so Python can find app.py
code_paths = list(Path("/kaggle/input").rglob("app.py"))
if code_paths:
    sys.path.append(str(code_paths[0].parent))

from app import process_video

# 2. Automatically locate any video file (.mp4, .avi, .mov, .mkv) in /kaggle/input
valid_exts = ("*.mp4", "*.avi", "*.mov", "*.mkv")
found_videos = []

for ext in valid_exts:
    found_videos.extend(list(Path("/kaggle/input").rglob(ext)))

if not found_videos:
    print("No video files found! Ensure your video dataset is added under '+ Add Input'.")
else:
    video_file = found_videos[0]
    print(f"Selected video for inference: {video_file}")
    
    # 3. Run the ChakraModel inference tracker
    out_video_path, df_timeline = process_video(
        str(video_file), 
        use_persistence=True, 
        window_size=10, 
        persistence_threshold=0.6, 
        doubt_policy="Warn Only", 
        imgsz_val="1024"
    )
    
    # 4. Save result to Kaggle output workspace
    output_dir = Path("/kaggle/working")
    final_output = output_dir / f"output_{video_file.name}"
    
    shutil.copy2(out_video_path, final_output)
    print(f"\nProcessing Complete! Saved to: {final_output}")
    print("\nTimeline of Detections:")
    display(df_timeline)
    

import inspect
import sys
from pathlib import Path

# Add directory containing app.py
code_path = Path("/kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/src")
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))

import app

# Print exported functions/classes
print("Functions available in app.py:")
print([item for item in dir(app) if not item.startswith("__")])

In [10]:
import sys
import shutil
from pathlib import Path

# 1. Create a writable directory in /kaggle/working
working_src = Path("/kaggle/working/src")
working_src.mkdir(parents=True, exist_ok=True)

# 2. Copy src contents from read-only input directory
orig_src = Path("/kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/src")
if orig_src.exists():
    for file in orig_src.glob("*.py"):
        shutil.copy2(file, working_src / file.name)

# 3. Insert /kaggle/working at the top of sys.path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# 4. Clear cached modules to ensure new imports load from /kaggle/working
for mod in ["app", "src.app"]:
    if mod in sys.modules:
        del sys.modules[mod]

print("✅ Directory copied to /kaggle/working/src successfully!")

✅ Directory copied to /kaggle/working/src successfully!


In [5]:
import sys
import shutil
from pathlib import Path

# Set paths
working_src = Path("/kaggle/working/src")
orig_src = Path("/kaggle/input/datasets/gokulrocky/chakramodel/kaggle_bundle/src")

# 1. Clean up old partial copy if it exists
if working_src.exists():
    shutil.rmtree(working_src)

# 2. Copy the ENTIRE src folder recursively (including subdirectories like temporal)
if orig_src.exists():
    shutil.copytree(orig_src, working_src)
    print("✅ Entire 'src' folder (including subdirectories) copied successfully!")

# 3. Add /kaggle/working to Python path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# 4. Clear any stale module caches
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod == "app":
        del sys.modules[mod]

✅ Entire 'src' folder (including subdirectories) copied successfully!


In [6]:
import sys
# Ensure /kaggle/working is at the top of sys.path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# Verify key imports
try:
    import app
    import src.temporal
    import src.benchmark_kvasir as bmk
    print("✅ All imports (app, src.temporal, src.benchmark_kvasir) loaded successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")

✅ All imports (app, src.temporal, src.benchmark_kvasir) loaded successfully!


In [8]:
import sys
from pathlib import Path

# Add working directory to sys.path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from src.download_kvasir import download_kvasir_seg
import src.benchmark_kvasir as bmk

working_dir = Path("/kaggle/working")
out_dir = working_dir / "outputs" / "eval_kaggle"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Download the dataset to /kaggle/working/data/kvasir-seg
print("⬇️ Downloading Kvasir-SEG dataset...")
dataset_path = download_kvasir_seg(working_dir)
print(f"✅ Dataset ready at: {dataset_path}")

# 2. Run the benchmark evaluation
print("🚀 Starting benchmark evaluation...")
metrics = bmk.run_benchmark("kvasir-seg", working_dir, out_dir)
print("✅ Benchmark metrics complete!")

⬇️ Downloading Kvasir-SEG dataset...
  Progress: 100.0%  (44MB / 44MB)
[EXTRACT] kvasir-seg.zip...
[OK] kvasir-seg ready: 1000 images
✅ Dataset ready at: /kaggle/working/data/kvasir-seg
🚀 Starting benchmark evaluation...

[BENCHMARK] ChakraModel PraNet on kvasir-seg
  Images: 1000
  Device: auto (CUDA if available)
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 160MB/s] 


[WARN] PraNetSegmenter: No weights found. Running with random initialization.
  [  50/1000] Running Dice: 0.3246 | 14.2ms/frame
  [ 100/1000] Running Dice: 0.2983 | 12.8ms/frame
  [ 150/1000] Running Dice: 0.3110 | 13.3ms/frame
  [ 200/1000] Running Dice: 0.3175 | 13.5ms/frame
  [ 250/1000] Running Dice: 0.3248 | 13.7ms/frame
  [ 300/1000] Running Dice: 0.3252 | 13.1ms/frame
  [ 350/1000] Running Dice: 0.3262 | 14.1ms/frame
  [ 400/1000] Running Dice: 0.3252 | 13.7ms/frame
  [ 450/1000] Running Dice: 0.3264 | 13.3ms/frame
  [ 500/1000] Running Dice: 0.3267 | 12.4ms/frame
  [ 550/1000] Running Dice: 0.3309 | 13.5ms/frame
  [ 600/1000] Running Dice: 0.3265 | 13.1ms/frame
  [ 650/1000] Running Dice: 0.3260 | 12.5ms/frame
  [ 700/1000] Running Dice: 0.3245 | 12.6ms/frame
  [ 750/1000] Running Dice: 0.3294 | 15.2ms/frame
  [ 800/1000] Running Dice: 0.3331 | 13.7ms/frame
  [ 850/1000] Running Dice: 0.3349 | 13.9ms/frame
  [ 900/1000] Running Dice: 0.3372 | 14.0ms/frame
  [ 950/1000] Running 

In [11]:
import sys
from pathlib import Path

# Point to working directory
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

import src.benchmark_kvasir as bmk

working_dir = Path("/kaggle/working")
out_dir = working_dir / "outputs" / "eval_kaggle"
out_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Starting benchmark evaluation...")
metrics = bmk.run_benchmark("kvasir-seg", working_dir, out_dir)
print("Benchmark metrics complete!")

🚀 Starting benchmark evaluation...

[BENCHMARK] ChakraModel PraNet on kvasir-seg
  Images: 1000
  Device: auto (CUDA if available)
[WARN] PraNetSegmenter: No weights found. Running with random initialization.
  [  50/1000] Running Dice: 0.3246 | 14.5ms/frame
  [ 100/1000] Running Dice: 0.2983 | 12.4ms/frame
  [ 150/1000] Running Dice: 0.3110 | 13.5ms/frame
  [ 200/1000] Running Dice: 0.3175 | 15.3ms/frame
  [ 250/1000] Running Dice: 0.3248 | 15.0ms/frame
  [ 300/1000] Running Dice: 0.3252 | 12.8ms/frame
  [ 350/1000] Running Dice: 0.3262 | 13.6ms/frame
  [ 400/1000] Running Dice: 0.3252 | 13.5ms/frame
  [ 450/1000] Running Dice: 0.3264 | 13.7ms/frame
  [ 500/1000] Running Dice: 0.3267 | 13.0ms/frame
  [ 550/1000] Running Dice: 0.3309 | 14.6ms/frame
  [ 600/1000] Running Dice: 0.3265 | 12.5ms/frame
  [ 650/1000] Running Dice: 0.3260 | 12.8ms/frame
  [ 700/1000] Running Dice: 0.3245 | 13.0ms/frame
  [ 750/1000] Running Dice: 0.3294 | 13.5ms/frame
  [ 800/1000] Running Dice: 0.3331 | 14.0